In [ ]:
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")


def split_into_utterances(df):
    utterances = []
    current_speaker = df.iloc[0]['speaker']
    current_indices = []

    for idx, row in df.iterrows():
        if row['speaker'] != current_speaker:
            # save current utterance
            utterances.append((current_speaker, current_indices))
            # start new
            current_speaker = row['speaker']
            current_indices = [idx]
        else:
            current_indices.append(idx)

    # append last one
    if current_indices:
        utterances.append((current_speaker, current_indices))

    return utterances


def extract_phrase_words_mask(df, indices, phrase_type='NP'):
    """
    Create a boolean mask for rows in df[indices] that belong to the specified phrase type.
    """
    utterance_words = df.loc[indices, 'word'].tolist()
    utterance_text = " ".join(utterance_words)
    doc = nlp(utterance_text)

    # Initialize mask
    mask = [False] * len(indices)

    if phrase_type == 'NP':
        np_words = set()
        for np in doc.noun_chunks:
            np_words.update([t.text for t in np])
        # mark mask
        mask = [word in np_words for word in utterance_words]

    elif phrase_type == 'VP':
        vp_words = set()
        for token in doc:
            if token.pos_ == "VERB":
                vp_words.add(token.text)
                for child in token.children:
                    if child.dep_ in ("aux", "auxpass", "advmod", "xcomp", "compound"):
                        vp_words.add(child.text)
        mask = [word in vp_words for word in utterance_words]

    return mask


def filter_df_by_phrase_mask(df, phrase_type='NP'):
    """
    Filter the original DataFrame by NP or VP using a boolean mask.
    """
    mask = [False] * len(df)
    utterances = split_into_utterances(df)

    for speaker, indices in utterances:
        utter_mask = extract_phrase_words_mask(df, indices, phrase_type)
        for i, m in zip(indices, utter_mask):
            mask[i] = m

    return df[mask]

In [ ]:
utterance_text = "This is a sample utterance for testing."
# utterance_text = "Skynet will be decommisioned soon."
utterance_text = "The machine that kills life will be decommissioned soon."
utterance_text = "I really don't like what you are suggesting."
utterance_words = utterance_text.split()

nlp = spacy.load("en_core_web_sm")
doc = nlp(utterance_text)
# np_words = set()
# for np in doc.noun_chunks:
#     np_words.update([t.text for t in np])
# # mark mask
# mask = [word in np_words for word in utterance_words]
vp_words = set()
for token in doc:
    print(token.pos_, token.dep_, token.text)
#     if token.pos_ == "VERB":
#         vp_words.add(token.text)
#         for child in token.children:
#             if child.dep_ in ("aux", "auxpass", "advmod", "xcomp", "compound"):
#                 vp_words.add(child.text)
# mask = [word in vp_words for word in utterance_words]
for sent in doc.sents:
    print(sent.root)

In [ ]:
import stanza
from nltk.tree import Tree

utterance_text = "This is a sample utterance for testing."
utterance_text = "Skynet will be decommisioned soon."
# utterance_text = "The machine that kills life will be decommissioned soon."
utterance_text = "One two three four five."
utterance_text = "How is this possible?"
utterance_text = "Wait, how is this possible?"
utterance_text = "I really don't like what you are suggesting."
utterance_text = "Woah that is amazing! I like it!"

nlp = stanza.Pipeline(lang='en', processors='tokenize,pos,constituency')
doc = nlp(utterance_text)
# for sentence in doc.sentences:
#     print(sentence.constituency)

df = pd.DataFrame()
sent_cons = doc.sentences[0].constituency.children[0]
for sent_con in sent_cons.children:
    nltk_tree = Tree.fromstring(str(sent_con))
    phrase_df = pd.DataFrame()
    phrase_df["words"] = nltk_tree.leaves()
    phrase_df["label"] = sent_con.label
    df = pd.concat([df, phrase_df])

print(df)
nltk_tree = Tree.fromstring(str(sent_cons))
nltk_tree.pretty_print()